# MIMIC-III Clinical Mortality Prediction: GNN + Symbolic AI Hybrid

**Author:** Nihant  
**Dataset:** [MIMIC-III Clinical Database Demo v1.4](https://physionet.org/content/mimiciii-demo/1.4/) (Kaggle)  
**Platform:** Kaggle Notebooks (GPU optional)  

---

## 🎯 Objective

Predict **in-hospital mortality** for ICU patients by fusing two complementary AI paradigms:

| Component | Role |
|---|---|
| **Graph Neural Network (GCN)** | Learns latent patient similarity embeddings from the population graph |
| **Symbolic Prolog Rules** | Encodes interpretable clinical domain knowledge (renal, sepsis, metabolic risk) |
| **Hybrid Fusion Classifier** | Combines both signal types for final mortality prediction |

## 📐 Architecture Overview

```
MIMIC-III Tables
     │
     ▼
Feature Engineering  ──►  Patient Population Graph
(labs + ICD-9 codes)            (cosine similarity edges)
                                     │
                                     ▼
                             2-Layer GCN
                          (GCNConv → GCNConv)
                                     │
                                32-dim Embeddings
                                     │
             ┌───────────────────────┘
             │
  Prolog Rule Engine  ──►  5 Clinical Rule Scores
  (renal / sepsis /              │
   metabolic risk)               ▼
                         Hybrid Feature Vector (37-dim)
                                  │
                    ┌─────────────┴──────────────┐
                    ▼                            ▼
          Logistic Regression         Gradient Boosting
            (interpretable)            (high performance)
```

## 📂 MIMIC-III Tables Used

| Table | Key Columns |
|---|---|
| `ADMISSIONS` | `hadm_id`, `hospital_expire_flag`, `ethnicity`, `insurance` |
| `PATIENTS` | `subject_id`, `gender`, `dob` |
| `DIAGNOSES_ICD` | `icd9_code` (top-20 codes as binary features) |
| `LABEVENTS` | creatinine (50912), WBC (51301), glucose (50931) |

## ⚙️ Cell 1 — Environment Setup

Install PyTorch Geometric and verify all library versions.  
> **Kaggle users:** Enable **Internet** in the notebook settings before running. Takes ~2–3 min.

In [1]:
import subprocess, sys

# Install PyTorch Geometric (stable build via pip)
subprocess.run([sys.executable, "-m", "pip", "install", "torch_geometric", "-q"], check=True)

import numpy as np
import pandas as pd
import torch
import torch_geometric
import sklearn

print(f"PyTorch:           {torch.__version__}")
print(f"PyTorch Geometric: {torch_geometric.__version__}")
print(f"scikit-learn:      {sklearn.__version__}")
print(f"CUDA available:    {torch.cuda.is_available()}")
print("✅ Environment ready!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.9 MB/s eta 0:00:00
PyTorch:           2.10.0+cu128
PyTorch Geometric: 2.7.0
scikit-learn:      1.6.1
CUDA available:    True
✅ Environment ready!


## ⚙️ Cell 2 — Global Configuration

Centralise all hyper-parameters and paths in one place for easy experimentation.

In [2]:
import os

# ─── Paths ────────────────────────────────────────────────────────────────────
BASE = (
    "/kaggle/input/datasets/atamazian/"
    "mimic-iii-clinical-dataset-demo/"
    "mimic-iii-clinical-database-demo-1.4/"
)

# ─── Graph construction ───────────────────────────────────────────────────────
SIMILARITY_THRESHOLD = 0.75   # cosine-sim edge threshold

# ─── GNN hyper-parameters ─────────────────────────────────────────────────────
GNN_HIDDEN_DIM  = 64
GNN_EMBED_DIM   = 32
GNN_DROPOUT     = 0.3
GNN_LR          = 0.01
GNN_WEIGHT_DECAY = 5e-4
GNN_EPOCHS      = 150
GNN_TRAIN_RATIO = 0.80
POS_CLASS_WEIGHT = 8.0        # up-weight minority (expired) class

# ─── Classifier hyper-parameters ──────────────────────────────────────────────
TEST_SIZE   = 0.20
RANDOM_SEED = 42
CV_FOLDS    = 5

# ─── Lab item IDs ─────────────────────────────────────────────────────────────
KEY_LABS = {50912: "creatinine", 51301: "wbc", 50931: "glucose"}
TOP_ICD_N = 20

# ─── Clinical thresholds (mirrors Prolog rules) ───────────────────────────────
CREATININE_HIGH = 2.0    # mg/dL
WBC_HIGH        = 12.0   # 10³/µL
WBC_LOW         = 4.0
GLUCOSE_HIGH    = 200.0  # mg/dL

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Compute device: {DEVICE}")

Compute device: cuda


## 📦 Cell 3 — Data Loading & Feature Engineering

We load four MIMIC-III tables and produce a flat patient-level DataFrame:

- **Lab pivot** – mean creatinine / WBC / glucose per admission  
- **ICD-9 binary flags** – top-N diagnosis codes as one-hot columns  
- **Mortality label** – `hospital_expire_flag` cast to int (0 = survived, 1 = expired)  
- **Missing lab imputation** – median fill (safe for small datasets)

In [3]:
# ─── Load tables ──────────────────────────────────────────────────────────────
admissions = pd.read_csv(
    BASE + "ADMISSIONS.csv",
    usecols=["subject_id","hadm_id","admittime","dischtime",
             "hospital_expire_flag","ethnicity","insurance"],
)
diagnoses = pd.read_csv(
    BASE + "DIAGNOSES_ICD.csv",
    usecols=["subject_id","hadm_id","icd9_code"],
)
labs = pd.read_csv(
    BASE + "LABEVENTS.csv",
    usecols=["subject_id","hadm_id","itemid","valuenum"],
)
patients = pd.read_csv(
    BASE + "PATIENTS.csv",
    usecols=["subject_id","gender","dob"],
)

# Normalise column names to lowercase for consistency
for tbl in [admissions, diagnoses, labs, patients]:
    tbl.columns = tbl.columns.str.lower()

# ─── Mortality label ──────────────────────────────────────────────────────────
admissions["mortality"] = admissions["hospital_expire_flag"].astype(int)

# ─── Lab features (mean per admission) ───────────────────────────────────────
labs_filtered = labs[labs["itemid"].isin(KEY_LABS)].copy()
labs_filtered["lab_name"] = labs_filtered["itemid"].map(KEY_LABS)

lab_pivot = (
    labs_filtered
    .groupby(["hadm_id", "lab_name"])["valuenum"]
    .mean()
    .unstack()
    .reset_index()
)

# ─── Merge into patient-level DataFrame ──────────────────────────────────────
df = (
    admissions
    .merge(lab_pivot, on="hadm_id", how="left")
    .merge(patients,  on="subject_id", how="left")
)

df["gender_m"] = (df["gender"] == "M").astype(int)

# Median imputation for missing lab values
for col in KEY_LABS.values():
    df[col] = df[col].fillna(df[col].median())

# ─── ICD-9 binary flags ───────────────────────────────────────────────────────
top_icd = diagnoses["icd9_code"].value_counts().head(TOP_ICD_N).index.tolist()
icd_hadm = {code: set(diagnoses[diagnoses["icd9_code"]==code]["hadm_id"]) for code in top_icd}

for code, hadm_set in icd_hadm.items():
    df[f"icd_{code}"] = df["hadm_id"].isin(hadm_set).astype(int)

df = df.dropna(subset=["mortality"]).reset_index(drop=True)

# ─── Summary ─────────────────────────────────────────────────────────────────
print(f"✅ Dataset ready: {len(df)} admissions")
print(f"   Mortality rate: {df['mortality'].mean():.2%}")
icd_cols = [c for c in df.columns if c.startswith('icd_')]
print(f"   ICD-9 features: {len(icd_cols)} codes")
df[["creatinine", "wbc", "glucose", "mortality"]].describe()

✅ Dataset ready: 129 admissions
   Mortality rate: 31.01%
   ICD-9 features: 20 codes


,creatinine,wbc,glucose,mortality
count,129.000000,129.000000,129.000000,129.000000
mean,1.348711,11.341251,141.051292,0.310078
std,1.350400,7.062569,44.757805,0.464328
min,0.335000,0.975000,78.363636,0.000000
25%,0.680000,7.285714,111.416667,0.000000
50%,0.960000,10.062500,130.074074,0.000000
75%,1.311111,13.675000,155.000000,1.000000
max,10.083333,65.000000,344.000000,1.000000


## 🕸️ Cell 4 — Patient Population Graph Construction

We model the ICU cohort as an **undirected patient similarity graph**:

- **Nodes** – individual patient admissions, each with a feature vector of labs + ICD flags
- **Edges** – drawn between pairs whose feature vectors share cosine similarity > `SIMILARITY_THRESHOLD`

> **Why cosine similarity?** It is scale-invariant and robust to the mix of continuous lab values and binary ICD flags.

In [4]:
from torch_geometric.data import Data
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

# ─── Feature matrix ───────────────────────────────────────────────────────────
feat_cols = (
    ["creatinine", "wbc", "glucose", "gender_m"]
    + [c for c in df.columns if c.startswith("icd_")]
)

X_raw = df[feat_cols].values.astype(np.float32)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

labels = df["mortality"].values

# ─── Build edges via cosine similarity threshold ──────────────────────────────
sim_matrix = cosine_similarity(X_scaled)
np.fill_diagonal(sim_matrix, 0)   # exclude self-loops
src, dst = np.where(sim_matrix > SIMILARITY_THRESHOLD)

edge_index = torch.tensor(np.stack([src, dst]), dtype=torch.long)
x_tensor   = torch.tensor(X_scaled, dtype=torch.float)
y_tensor   = torch.tensor(labels,   dtype=torch.long)

data = Data(x=x_tensor, edge_index=edge_index, y=y_tensor)

avg_degree = data.num_edges / data.num_nodes
print(f"✅ Patient graph built:")
print(f"   Nodes (admissions):  {data.num_nodes}")
print(f"   Edges (pairs):       {data.num_edges}")
print(f"   Avg node degree:     {avg_degree:.1f}")
print(f"   Feature dims:        {data.num_features}")
print(f"   Positive (expired):  {int(y_tensor.sum())} / {len(y_tensor)} "
      f"({y_tensor.float().mean():.1%})")

✅ Patient graph built:
   Nodes (admissions):  129
   Edges (pairs):       48
   Avg node degree:     0.4
   Feature dims:        24
   Positive (expired):  40 / 129 (31.0%)


## 🧠 Cell 5 — Graph Neural Network: Architecture & Training

We use a **2-layer Graph Convolutional Network (GCN)**:

```
Input (F features)  →  GCNConv(F, 64)  →  ReLU  →  Dropout(0.3)
                    →  GCNConv(64, 32)  →  Linear(32, 2)  →  logits
```

**Key training choices:**

| Decision | Rationale |
|---|---|
| `class_weight = [1.0, 8.0]` | ~10% positive class — up-weight minority to avoid trivial 'all-survived' predictions |
| `weight_decay = 5e-4` | L2 regularisation on a tiny dataset (129 patients) |
| `Dropout(0.3)` | Reduces overfitting between the two GCN layers |
| `seed=42` for masks | Reproducible 80/20 train-val split |

In [5]:
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

# ─── Model definition ─────────────────────────────────────────────────────────
class PatientGNN(torch.nn.Module):
    """Two-layer GCN that returns both class logits and node embeddings."""

    def __init__(self, in_channels: int, hidden: int = GNN_HIDDEN_DIM, emb: int = GNN_EMBED_DIM):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden)
        self.conv2 = GCNConv(hidden, emb)
        self.classifier = torch.nn.Linear(emb, 2)

    def forward(self, x, edge_index):
        emb = F.relu(self.conv1(x, edge_index))
        emb = F.dropout(emb, p=GNN_DROPOUT, training=self.training)
        emb = self.conv2(emb, edge_index)
        logits = self.classifier(emb)
        return logits, emb   # (N, 2) and (N, 32)

# ─── Initialise ───────────────────────────────────────────────────────────────
model     = PatientGNN(data.num_features).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=GNN_LR, weight_decay=GNN_WEIGHT_DECAY)
data_d    = data.to(DEVICE)

# ─── Train / val masks (reproducible) ────────────────────────────────────────
n    = data_d.num_nodes
perm = torch.randperm(n, generator=torch.Generator().manual_seed(RANDOM_SEED))
train_mask = torch.zeros(n, dtype=torch.bool, device=DEVICE)
val_mask   = torch.zeros(n, dtype=torch.bool, device=DEVICE)
train_mask[perm[:int(GNN_TRAIN_RATIO * n)]] = True
val_mask[perm[int(GNN_TRAIN_RATIO * n):]]   = True

# ─── Class weights (address class imbalance) ──────────────────────────────────
pos_weight = torch.tensor([1.0, POS_CLASS_WEIGHT], device=DEVICE)

print(f"Training on: {DEVICE}")
print(f"Train nodes: {train_mask.sum().item()} | Val nodes: {val_mask.sum().item()}")
print("-" * 40)

# ─── Training loop ────────────────────────────────────────────────────────────
model.train()
for epoch in range(1, GNN_EPOCHS + 1):
    optimizer.zero_grad()
    logits, _ = model(data_d.x, data_d.edge_index)
    loss = F.cross_entropy(logits[train_mask], data_d.y[train_mask], weight=pos_weight)
    loss.backward()
    optimizer.step()

    if epoch % 30 == 0:
        # Quick val accuracy (not used for early stopping, just logging)
        model.eval()
        with torch.no_grad():
            val_logits, _ = model(data_d.x, data_d.edge_index)
            val_preds = val_logits[val_mask].argmax(dim=1)
            val_acc   = (val_preds == data_d.y[val_mask]).float().mean().item()
        model.train()
        print(f"  Epoch {epoch:3d} | Loss: {loss.item():.4f} | Val acc: {val_acc:.3f}")

# ─── Extract embeddings ───────────────────────────────────────────────────────
model.eval()
with torch.no_grad():
    _, gnn_emb = model(data_d.x, data_d.edge_index)
    gnn_embeddings = gnn_emb.cpu().numpy()

print(f"\n✅ GNN embeddings: {gnn_embeddings.shape}  (patients × 32-dim)")

Training on: cuda
Train nodes: 103 | Val nodes: 26
----------------------------------------
  Epoch  30 | Loss: 0.1332 | Val acc: 0.615
  Epoch  60 | Loss: 0.0756 | Val acc: 0.615
  Epoch  90 | Loss: 0.0721 | Val acc: 0.731
  Epoch 120 | Loss: 0.0601 | Val acc: 0.769
  Epoch 150 | Loss: 0.0879 | Val acc: 0.731

✅ GNN embeddings: (129, 32)  (patients × 32-dim)


## 📋 Cell 6 — Symbolic Clinical Rules (Vectorised Prolog)

We encode five ICU clinical risk rules, originally expressed in Prolog, as vectorised pandas operations.  
This approach is **orders of magnitude faster** than a Prolog interpreter at inference time while producing identical outputs.

```prolog
renal_risk(P)     :- creatinine(P, Cr), Cr > 2.0.
sepsis_risk(P)    :- wbc(P, W), (W > 12.0 ; W < 4.0).
metabolic_risk(P) :- glucose(P, G), G > 200.
high_risk(P)      :- renal_risk(P), sepsis_risk(P).
high_risk(P)      :- renal_risk(P), metabolic_risk(P).
medium_risk(P)    :- sepsis_risk(P), metabolic_risk(P).
```

> These thresholds are drawn from established clinical guidelines (AKI: creatinine > 2.0 mg/dL; SIRS/Sepsis: WBC outside 4–12 × 10³/µL).

In [6]:
def apply_clinical_rules(df_: pd.DataFrame) -> pd.DataFrame:
    """
    Vectorised mirror of Prolog clinical risk rules.
    Returns a DataFrame of 5 binary rule-activation columns.
    """
    renal     = (df_["creatinine"] > CREATININE_HIGH).astype(int)
    sepsis    = ((df_["wbc"] > WBC_HIGH) | (df_["wbc"] < WBC_LOW)).astype(int)
    metabolic = (df_["glucose"] > GLUCOSE_HIGH).astype(int)
    high      = ((renal & sepsis) | (renal & metabolic)).astype(int)
    medium    = (sepsis & metabolic).astype(int)

    return pd.DataFrame({
        "rule_renal"      : renal.values,
        "rule_sepsis"     : sepsis.values,
        "rule_metabolic"  : metabolic.values,
        "rule_high_risk"  : high.values,
        "rule_medium_risk": medium.values,
    })


prolog_scores = apply_clinical_rules(df)

print("✅ Rule activation counts:")
print(prolog_scores.sum().to_string())
print(f"\nHigh-risk flagged:   {prolog_scores['rule_high_risk'].sum()} patients")
print(f"Medium-risk flagged: {prolog_scores['rule_medium_risk'].sum()} patients")

✅ Rule activation counts:
rule_renal          18
rule_sepsis         49
rule_metabolic      12
rule_high_risk       6
rule_medium_risk     9

High-risk flagged:   6 patients
Medium-risk flagged: 9 patients


## 🔀 Cell 7 — Hybrid Fusion Classifier

We concatenate the **32-dim GNN embeddings** with **5 Prolog rule scores** to create a 37-dimensional hybrid feature vector, then train two classifiers:

| Classifier | Strength |
|---|---|
| **Logistic Regression** | Interpretable, fast, calibrated probabilities |
| **Gradient Boosting** | Non-linear, captures feature interactions |

Both use `StratifiedKFold` cross-validation (k=5) to account for class imbalance in this small cohort.

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ─── Fuse GNN embeddings + Prolog rule scores ─────────────────────────────────
prolog_arr   = prolog_scores.values.astype(np.float32)
hybrid_feats = np.hstack([gnn_embeddings, prolog_arr])   # (N, 37)
y            = df["mortality"].values

print(f"Hybrid feature shape: {hybrid_feats.shape}")
print(f"  └── {gnn_embeddings.shape[1]} GNN dims + {prolog_arr.shape[1]} Prolog rules")

# ─── Stratified train / test split ───────────────────────────────────────────
X_tr, X_te, y_tr, y_te = train_test_split(
    hybrid_feats, y,
    test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
)

print(f"\nTrain size: {len(X_tr)} | Test size: {len(X_te)}")
print(f"Train mortality rate: {y_tr.mean():.2%} | Test: {y_te.mean():.2%}")

# ─── Pipelines ───────────────────────────────────────────────────────────────
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        max_iter=1000, class_weight="balanced", C=0.5, random_state=RANDOM_SEED
    )),
])

gb_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", GradientBoostingClassifier(
        n_estimators=100, max_depth=3, learning_rate=0.1,
        subsample=0.8, random_state=RANDOM_SEED
    )),
])

lr_pipe.fit(X_tr, y_tr)
gb_pipe.fit(X_tr, y_tr)

# ─── 5-fold cross-validation ──────────────────────────────────────────────────
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)
lr_cv = cross_val_score(lr_pipe, hybrid_feats, y, cv=cv, scoring="f1_macro")
gb_cv = cross_val_score(gb_pipe, hybrid_feats, y, cv=cv, scoring="f1_macro")

print(f"\n✅ {CV_FOLDS}-Fold Cross-Validation F1 (macro):")
print(f"   Logistic Regression: {lr_cv.mean():.4f} ± {lr_cv.std():.4f}")
print(f"   Gradient Boosting:   {gb_cv.mean():.4f} ± {gb_cv.std():.4f}")

Hybrid feature shape: (129, 37)
  └── 32 GNN dims + 5 Prolog rules

Train size: 103 | Test size: 26
Train mortality rate: 31.07% | Test: 30.77%

✅ 5-Fold Cross-Validation F1 (macro):
   Logistic Regression: 0.8701 ± 0.0461
   Gradient Boosting:   0.8907 ± 0.0576


## 📊 Cell 8 — Full Evaluation & Ablation Study

We evaluate on the held-out test set with:
- **F1 macro** – balanced metric for imbalanced classes
- **F1 (positive)** – specifically for the minority 'expired' class
- **ROC-AUC** – discriminative power across all thresholds
- **Confusion matrix** – raw prediction breakdown

The **ablation study** quantifies how much each component (GNN alone, Prolog alone, or combined) contributes to overall performance.

In [8]:
from sklearn.metrics import (
    f1_score, classification_report, roc_auc_score, confusion_matrix
)

def evaluate_model(pipe, X_te, y_te, name: str) -> float:
    """Print a full evaluation report and return macro-F1."""
    y_pred  = pipe.predict(X_te)
    y_proba = pipe.predict_proba(X_te)[:, 1]

    f1_mac  = f1_score(y_te, y_pred, average="macro")
    f1_bin  = f1_score(y_te, y_pred, average="binary")
    auc     = roc_auc_score(y_te, y_proba)
    cm      = confusion_matrix(y_te, y_pred)

    print(f"\n{'═'*55}")
    print(f"  {name}")
    print(f"{'═'*55}")
    print(f"  F1 macro:           {f1_mac:.4f}")
    print(f"  F1 (positive class):{f1_bin:.4f}")
    print(f"  ROC-AUC:            {auc:.4f}")
    print(f"  Confusion matrix:\n  {cm}")
    print(classification_report(y_te, y_pred, target_names=["Survived", "Expired"]))
    return f1_mac


evaluate_model(lr_pipe, X_te, y_te, "Logistic Regression · Hybrid (GNN + Prolog)")
evaluate_model(gb_pipe, X_te, y_te, "Gradient Boosting   · Hybrid (GNN + Prolog)")

# ─── Ablation study ───────────────────────────────────────────────────────────
print("\n" + "─" * 55)
print("  Ablation: Contribution of each component")
print("─" * 55)

def quick_f1(X_all, y_all, label: str, feat_slice):
    """Train a simple LR on a feature subset and print bar-chart F1."""
    Xa, Xb, ya, yb = train_test_split(
        X_all[:, feat_slice], y_all,
        test_size=TEST_SIZE, stratify=y_all, random_state=RANDOM_SEED
    )
    pipe = Pipeline([
        ("sc", StandardScaler()),
        ("m",  LogisticRegression(max_iter=500, class_weight="balanced", random_state=RANDOM_SEED)),
    ])
    pipe.fit(Xa, ya)
    f1  = f1_score(yb, pipe.predict(Xb), average="macro")
    bar = "█" * int(f1 * 40)
    print(f"  {label:35s} {bar:40s}  {f1:.4f}")


quick_f1(hybrid_feats, y, "GNN only        (32 features)", slice(None, 32))
quick_f1(hybrid_feats, y, "Prolog only     ( 5 rules)   ", slice(32, None))
quick_f1(hybrid_feats, y, "Hybrid ✅        (GNN + Prolog)", slice(None))


═══════════════════════════════════════════════════════
  Logistic Regression · Hybrid (GNN + Prolog)
═══════════════════════════════════════════════════════
  F1 macro:           0.8462
  F1 (positive class):0.7692
  ROC-AUC:            0.9792
  Confusion matrix:
  [[18  0]
 [ 3  5]]
              precision    recall  f1-score   support

    Survived       0.86      1.00      0.92        18
     Expired       1.00      0.62      0.77         8

    accuracy                           0.88        26
   macro avg       0.93      0.81      0.85        26
weighted avg       0.90      0.88      0.88        26


═══════════════════════════════════════════════════════
  Gradient Boosting   · Hybrid (GNN + Prolog)
═══════════════════════════════════════════════════════
  F1 macro:           0.7436
  F1 (positive class):0.6154
  ROC-AUC:            0.9167
  Confusion matrix:
  [[17  1]
 [ 4  4]]
              precision    recall  f1-score   support

    Survived       0.81      0.94      0.87 

---

## 🏁 Summary & Next Steps

### What we built

A **Neuro-Symbolic hybrid pipeline** for ICU mortality prediction that fuses:
1. **GCN embeddings** — data-driven patient similarity representations
2. **Prolog-derived clinical rules** — interpretable domain knowledge signals

The ablation study validates that the **hybrid outperforms either component alone**, demonstrating the complementary nature of learned and symbolic representations.

### Potential extensions

| Extension | Description |
|---|---|
| **Temporal GNN** | Replace static snapshot with time-aware message passing (e.g., T-GCN) |
| **Full MIMIC-III** | Scale to ~46k admissions using PyG's `NeighborLoader` for mini-batch training |
| **SHAP explanations** | Quantify feature importance across the hybrid feature space |
| **Real PyProlog integration** | Use actual Prolog inference for richer, recursive rule chains |
| **Attention GNN** | Replace GCNConv with GATConv for interpretable edge weights |

---
*Notebook generated for educational and research purposes. MIMIC-III data requires credentialed access via PhysioNet.*